# Evaluating an AI Agent with RAGAS and DeepEval (Annotated)

> **About this notebook:** This is an annotated copy of `agentic_system_eval.ipynb`. **The code is the same as the original.** Only `#` comments and explanation cells have been added so the code is easier to learn from. The saved outputs are from the original run.

---

## 1. The big picture in plain English

Picture a new **travel-agent intern**. You give them three tools: a flight search screen, a hotel booking screen and a weather website. A customer says *"Find me flights from Bangalore to Dubai and a 4-star hotel there."*

A good intern would:
1. **Pick the right tools**: open flight search and hotel search, and leave the weather site alone.
2. **Type the right details** into those tools: `Bangalore → Dubai` and `4 stars, Dubai`.
3. **Give the customer a useful final answer.**

This notebook builds that intern as an **AI agent**, an LLM that can call Python functions ("tools"). Then it **grades** the intern on those three points, automatically, using two popular evaluation libraries:

| Library | Section in this notebook |
|---|---|
| **RAGAS** | Part A (first half) |
| **DeepEval** | Part B (second half, after the "Deepeval agent Evaluation" heading) |

Both parts use **the same agent, the same tools and the same three test questions**, so you can compare how the two libraries score the same behaviour.

---

## 2. Why evaluate agents at all?

A normal chatbot only produces text. An **agent** also takes *actions* (tool calls). It can fail in ways a chatbot can't:

| Failure | Example |
|---|---|
| Wrong tool | Calls `get_weather` when the user asked for hotels |
| Missing tool | Finds flights but forgets the hotel |
| Extra or unnecessary tool | Checks the weather when nobody asked |
| Wrong arguments | `find_hotels(city="Dubai", stars=5)` when the user said 4-star |
| Bad final answer | Calls the right tools but gives a confusing or incomplete reply |

Reading answers by hand doesn't scale, so we write a small **test dataset** (question → expected tool calls → expected goal) and let metrics score the agent.

---

## 3. How the code flows

```
             ┌─────────────────────────────────────────────┐
 Question ─► │  AGENT  (LLM + 3 tools, built with LangChain) │
             └─────────────────────────────────────────────┘
                        │ messages: Human → AI(tool_calls) → Tool results → AI(final answer)
                        ▼
             ┌───────────────────────────┐
             │ run_agent()               │  pulls out: final text answer
             │                           │             + list of tool calls made
             └───────────────────────────┘
                        │
                        ▼
             ┌───────────────────────────┐        ┌──────────────────────────────┐
             │ evaluate_test_case()      │ ◄───── │ dataset: expected tool calls │
             │  runs 3 metrics           │        │          + expected goal     │
             └───────────────────────────┘        └──────────────────────────────┘
                        │
                        ▼
             run_evaluation() loops over all test cases
                        │
                        ▼
             main() → prints a table, averages, per-case details, saves CSV
```

The same pipeline is written **twice**: once with RAGAS objects and once with DeepEval objects.

---

## 4. The building blocks

### The tools (mock / fake on purpose)
`search_flights`, `find_hotels`, `get_weather` **don't call real APIs**. They return hard-coded text. This is deliberate. It makes the experiment cheap and repeatable, and it lets us focus on **whether the agent calls the tools correctly**, not on whether some external API was up. (That's also why a "5-star" Mumbai hotel comes back as a Novotel: the data is fake.)

### The agent
`create_agent(...)` from LangChain wires an OpenAI chat model to the tools with a **system prompt** of rules ("use `get_weather` only when weather is requested", etc.). Internally the agent loops: *LLM decides → tool runs → LLM reads the result → … → final answer.*

### The test dataset (the "answer key")
Three questions, each with:
- the **reference tool calls** we expect (tool name + exact arguments)
- (RAGAS only) a **reference goal**: a sentence describing what a good answer must contain

| # | Question | Expected tool calls |
|---|---|---|
| 1 | Plan a Dubai trip from Bangalore (flights + 4-star hotel) | `search_flights(Bangalore, Dubai)`, `find_hotels(Dubai, 4)` |
| 2 | Find flights from Bangalore to Singapore | `search_flights(Bangalore, Singapore)` |
| 3 | Find a 5-star hotel in Mumbai | `find_hotels(Mumbai, 5)` |

---

## 5. The metrics explained

### Part A — RAGAS metrics

| Metric | Uses an LLM judge? | What it checks | Score |
|---|---|---|---|
| **ToolCallAccuracy** (`strict_order=False`) | No (rule-based) | Did the agent call the expected tools with **exactly** the expected arguments? Order doesn't matter here. | 0.0–1.0. Average argument match across expected calls; 0 if the set of tool names doesn't match. |
| **ToolCallF1** | No (rule-based) | Treats tool calls as a set and computes **precision** (how many calls made were correct?) and **recall** (how many expected calls were made?), then F1. Penalises *extra* calls as well as *missing* ones. | 0.0–1.0 |
| **AgentGoalAccuracyWithReference** | **Yes** (`gpt-5.6-terra`) | An LLM reads the conversation and compares the outcome to the `reference_goal` sentence. | **Binary**: 1.0 = goal achieved, 0.0 = not |

### Part B — DeepEval metrics

| Metric | Uses an LLM judge? | What it checks | Score |
|---|---|---|---|
| **ToolCorrectnessMetric** | No (rule-based) | Were all the **expected tools** called? (by name, order not considered here) | 0.0–1.0, pass if ≥ `threshold` 0.7 |
| **ArgumentCorrectnessMetric** | **Yes** (`gpt-5.4`) | Given the user's input, were the **arguments** passed to each tool sensible and correct? | 0.0–1.0 plus a written *reason* |
| **GoalAccuracyMetric** | **Yes** (`gpt-5.4`) | Looks at the conversation and scores **task completion** *and* **planning quality**, then combines them. | 0.0–1.0 plus a written *reason* |

> **Rule-based vs LLM-as-a-judge:** Rule-based metrics are free, fast and deterministic, but they only check exact matches. LLM-judge metrics understand meaning and give reasons, but they cost API calls, can vary between runs and depend on how strict the judge model is.

---

## 6. What the saved results show

| Metric | Average score (3 test cases) |
|---|---|
| RAGAS Tool Call Accuracy | **1.00** |
| RAGAS Tool Call F1 | **1.00** |
| RAGAS Agent Goal Accuracy | **1.00** |
| DeepEval Tool Correctness | **1.00** |
| DeepEval Argument Correctness | **1.00** |
| DeepEval Goal Accuracy | **0.58** (0.75, 0.375, 0.625) |

**What this tells us:** the agent picked the right tools with the right arguments every time, and both libraries agree on that. The difference is in **goal accuracy**. RAGAS compares the answer to our short reference goal and gives a yes/no, so it scores 1.0. DeepEval's judge is stricter. It also scores the *planning* and expects extras like travel dates and flight times, so it marks the answers down. **Same agent, different judges, different opinions.** That's why you should read the *reasons*, not just the numbers.

---

## 7. Prerequisites

- Python 3.11 environment with: `langchain`, `langchain-openai`, `openai`, `ragas` (0.4.x), `deepeval`, `pandas`, `python-dotenv`
- A `.env` file next to the notebook containing `OPENAI_API_KEY=sk-...`
- **Run cells top to bottom.** Part B **re-defines** `llm`, `agent`, `dataset`, `ToolCall`, `run_agent`, `evaluate_test_case` and `main` with DeepEval versions. If you run cells out of order, you may mix RAGAS and DeepEval objects by accident.
- Cells like `await main()` work because Jupyter already runs an *event loop*. In a normal `.py` script you'd write `asyncio.run(main())` instead.

---

## 8. Glossary (for beginners)

| Term | Meaning |
|---|---|
| **LLM** | Large Language Model, e.g. GPT-4o |
| **Agent** | An LLM that can decide to call functions (tools) and use their results |
| **Tool / tool call** | A Python function the LLM can request, plus the arguments it chose |
| **System prompt** | Hidden instructions that set the agent's behaviour |
| **Trace** | The full list of messages in one agent run (user, AI, tool results) |
| **Reference / ground truth** | The "answer key" we compare against |
| **LLM-as-a-judge** | Using a second LLM to grade the first LLM's output |
| **async / await** | Python syntax for running slow network calls without blocking; `await` waits for an `async` function to finish |
| **Precision / Recall / F1** | Precision = "of what I did, how much was right"; Recall = "of what I should have done, how much did I do"; F1 = their balanced average |

---

# Part A — Agent evaluation with RAGAS

### Step 0 — Sanity check
A trivial cell that confirms the Jupyter kernel is alive and running.

In [1]:
# Simple "is everything working?" check. If you see "all ok", the kernel is running.
print("all ok")

all ok


### Step 1 — Imports
This brings in everything the RAGAS half needs: the agent builder (LangChain), the OpenAI model wrappers, and the RAGAS metric classes and message types.

In [2]:
# --- Standard library ---------------------------------------------------------
import os        # read environment variables (e.g. the OpenAI API key)
import asyncio   # Python's async framework; the agent & RAGAS metrics are async

# --- Data handling -------------------------------------------------------------
import pandas as pd   # turn results into a table (DataFrame), compute averages, save CSV

# --- Config / secrets ----------------------------------------------------------
from dotenv import load_dotenv   # loads KEY=VALUE pairs from a .env file into os.environ

# --- OpenAI client (used by the RAGAS "judge" LLM) -----------------------------
from openai import AsyncOpenAI   # async version of the OpenAI SDK client

# --- LangChain: building the agent ---------------------------------------------
from langchain.agents import create_agent    # builds a tool-calling agent (LLM + tools + prompt)
from langchain_openai import ChatOpenAI      # LangChain wrapper around OpenAI chat models

# --- RAGAS: evaluation framework ------------------------------------------------
from ragas.llms import llm_factory           # wraps an LLM so RAGAS can use it as a judge

from ragas.messages import ToolCall          # RAGAS's way to describe a tool call: name + args

# The three agent metrics we will compute in Part A
from ragas.metrics.collections import (
    ToolCallAccuracy,                 # rule-based: right tools + exact right arguments?
    ToolCallF1,                       # rule-based: precision/recall over tool calls
    AgentGoalAccuracyWithReference,   # LLM-judge: did the agent achieve the reference goal?
)

from langchain_core.tools import tool        # decorator that turns a Python function into an LLM tool

# Converter from LangChain messages -> RAGAS messages
# (imported here but only used in a commented-out version of run_agent below)
from ragas.integrations.langgraph import convert_to_ragas_messages

/Users/manny/Documents/git_repos/Full-stack-GenAI/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2 — Load the API key
This reads secrets from a `.env` file so the key never has to be typed into the notebook. If it fails, the notebook stops early with a clear message.

In [3]:
# Read the .env file (in the current folder or a parent folder) and put its
# values into environment variables, e.g. OPENAI_API_KEY=sk-...
load_dotenv()

# Fail fast with a helpful message instead of a confusing error later on
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is missing. Please add it to the .env file."
    )

### Step 3 — Define the tools the agent can use
Each function below becomes a **tool** thanks to the `@tool` decorator. Two details matter a lot:
- The **docstring** (text in triple quotes) is sent to the LLM as the tool's description. The LLM reads it to decide *when* to use the tool.
- The **type hints** (`origin: str`, `stars: int`) become the tool's input schema. The LLM reads it to decide *what arguments* to pass.

The tools return **fake, hard-coded data**, so results are repeatable and the evaluation focuses on the agent's *decisions*.

In [4]:
# TOOL 1: flight search (MOCK, so it returns the same fake flights for any route).
# @tool converts this function into something the LLM can "call":
#   - function name  -> tool name the LLM sees ("search_flights")
#   - type hints     -> argument schema (origin: string, destination: string)
#   - docstring      -> tool description the LLM uses to decide WHEN to call it
@tool
def search_flights(origin: str, destination: str) -> str:
    """
    Search flights between an origin and destination city.
    """

    # Only the city names change; the flights/prices are hard-coded test data.
    # .strip() removes the leading/trailing newlines from the triple-quoted string.
    return f"""
Available flights from {origin} to {destination}:

1. Emirates EK569
   Price: ₹28,000
   Type: Non-stop

2. Air India AI995
   Price: ₹24,500
   Type: 1 stop
""".strip()

In [5]:
# TOOL 2: hotel search (MOCK).
# Note the `stars: int` hint: the LLM must pass a NUMBER (4), not text ("four").
# The evaluation later checks this argument exactly.
@tool
def find_hotels(city: str, stars: int) -> str:
    """
    Find hotels in a city based on star rating.
    """

    # Same two fake hotels for every city/star rating; only the names are filled in
    return f"""
Available {stars}-star hotels in {city}:

1. Novotel {city}
   Price: ₹8,500 per night

2. Hilton Garden Inn {city}
   Price: ₹9,200 per night
""".strip()

In [6]:
# TOOL 3: weather lookup (MOCK).
# None of our test questions ask about weather, so a well-behaved agent should
# NEVER call this tool. It's a "distractor" that tests whether the agent avoids
# unnecessary tool calls.
@tool
def get_weather(city: str) -> str:
    """
    Get weather information for a city.
    """

    return f"""
Weather in {city}:

Temperature: 30°C
Condition: Sunny
""".strip()


### Step 4 — Choose the LLM that powers the agent
`temperature=0` makes the model as deterministic as possible, which is what you want when testing. The commented-out blocks show other models the author tried.

In [7]:
# Alternative model configurations the author experimented with (disabled):

# llm = ChatOpenAI(
#     model="gpt-5.6-luna",
#     temperature=0,
# )

# Same model, but using OpenAI's newer "Responses API" message format:
# llm = ChatOpenAI(
#     model="gpt-5.6-luna",
#     use_responses_api=True,
#     output_version="responses/v1",
# )

# ACTIVE configuration: GPT-4o with temperature=0.
# temperature=0 -> the model picks the most likely output each time, which makes
# runs as repeatable as possible (important when you're testing!).
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
)


Quick connectivity test: send "hi" and confirm we get an `AIMessage` back.

In [8]:
# Smoke test: one direct call to the model (no tools, no agent).
# Returns an AIMessage. Check that `content` contains a greeting.
llm.invoke("hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_936db6157f', 'id': 'chatcmpl-EkXm1FrBQB7JlFQMn1ODBk1kMTCXE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e020-fddc-7dc1-ab61-665ea6640692-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Step 5 — Build the agent
`create_agent` combines **model + tools + system prompt** into a runnable agent. The system prompt's rules matter for evaluation. Rule 4 ("Do not call unnecessary tools") is exactly what **ToolCallF1** will penalise if broken, and rule 5 ("Use city names exactly") is what makes exact argument matching possible.

In [9]:
# Build the agent = LLM + list of tools + system prompt.
# Under the hood the agent runs a loop:
#   1. LLM reads the conversation and decides: answer directly OR call tool(s)
#   2. If tools were requested, LangChain runs the Python functions
#   3. Tool outputs are added to the conversation as ToolMessages
#   4. Repeat until the LLM writes a final answer with no tool calls
agent = create_agent(
    model=llm,

    # The tools the agent is ALLOWED to use
    tools=[
        search_flights,
        find_hotels,
        get_weather,
    ],

    # System prompt = standing instructions that shape the agent's behaviour.
    # These rules line up with what the metrics check later:
    #   rules 1-3 -> correct tool choice   (ToolCallAccuracy / ToolCorrectness)
    #   rule 4    -> no extra calls        (ToolCallF1 penalises extras)
    #   rule 5    -> exact arguments       (exact-match argument checks)
    #   rule 6    -> good final answer     (Goal Accuracy)
    system_prompt="""
You are a travel planning assistant.

Use the available tools whenever the user asks for
flight, hotel, or weather information.

Important rules:

1. Use search_flights for flight requests.
2. Use find_hotels for hotel requests.
3. Use get_weather only when weather is requested.
4. Do not call unnecessary tools.
5. Use city names exactly as provided by the user.
6. After using the tools, provide a clear final answer.
"""
)

### Step 6 — Try the agent once by hand
Look at the output. It's a list of messages, and it shows the agent's internal loop:
1. `HumanMessage`: the user's question
2. `AIMessage` with empty `content` but **`tool_calls=[search_flights(...), find_hotels(...)]`**: the LLM decided to call two tools (in parallel)
3. Two `ToolMessage`s: the outputs returned by our Python functions
4. Final `AIMessage`: the answer written for the user

The evaluation code later pulls out items 2 and 4.

In [10]:
# Run the agent once, by hand, to see what its output looks like.
# Input format: a dict with a "messages" list (chat-style role/content pairs).
# Output: a dict whose "messages" list holds the WHOLE trace:
#   HumanMessage -> AIMessage(tool_calls=[...]) -> ToolMessage(s) -> AIMessage(final answer)
agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),
        }
    ]
})

{'messages': [HumanMessage(content='Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.', additional_kwargs={}, response_metadata={}, id='b9ea2402-c82f-4335-9921-2c7d4fa11871'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 218, 'total_tokens': 273, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_be90e528c0', 'id': 'chatcmpl-EkXm14tuXA6qgbAbTlpa3NeBdlJFu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1e021-0072-7932-adeb-87b3b23a9bb9-0', tool_calls=[{'name': 'search_flights', 'args': {'origin': 'Bangalore', 'destination': 'Du

### Step 7 — Set up the "judge" LLM for RAGAS
Some RAGAS metrics need an LLM to grade the agent. We use a **different, separate model** (`gpt-5.6-terra`) as the judge. RAGAS's newer `metrics.collections` API is async, so it needs the **async** OpenAI client.

In [11]:
# Async OpenAI client: RAGAS "collections" metrics are async (they use `await`),
# so the judge LLM needs an async client.
openai_client = AsyncOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)

# The JUDGE model (separate from the agent's model).
# llm_factory wraps the client into the LLM interface RAGAS metrics expect.
# Only AgentGoalAccuracyWithReference uses it; the other two metrics are rule-based.
evaluator_llm = llm_factory(
    "gpt-5.6-terra",
    client=openai_client,
)

### Step 8 — Create the three RAGAS metrics
- `ToolCallAccuracy(strict_order=False)`: exact name and argument matching; order doesn't matter
- `ToolCallF1()`: set-based precision and recall over tool calls
- `AgentGoalAccuracyWithReference(llm=...)`: LLM judge, binary pass/fail against a reference goal

In [12]:
# METRIC 1: Tool Call Accuracy (rule-based, no LLM).
# Compares the agent's tool calls with the reference tool calls:
#   - the set of tool NAMES must match, otherwise score = 0
#   - then each call's ARGUMENTS are compared for an exact match
# strict_order=False -> calling find_hotels before search_flights is still OK.
tool_call_accuracy_metric = ToolCallAccuracy(strict_order=False)

In [13]:
# METRIC 2: Tool Call F1 (rule-based, no LLM).
# Treats tool calls as a set (a call matches only if name AND args match exactly):
#   precision = correct calls / calls the agent made      (penalises EXTRA calls)
#   recall    = correct calls / calls we expected         (penalises MISSING calls)
#   F1        = 2 * P * R / (P + R)
tool_call_f1_metric = ToolCallF1()

In [14]:
# METRIC 3: Agent Goal Accuracy WITH reference (LLM-as-a-judge).
# The judge LLM reads the conversation, works out what the agent achieved, and
# compares it with our written `reference_goal`. Binary score: 1.0 = achieved, 0.0 = not.
agent_goal_accuracy_metric = AgentGoalAccuracyWithReference(llm=evaluator_llm)

### Step 9 — The test dataset (our "answer key")
Each test case has a **question**, the **reference tool calls** we expect (name + exact arguments) and a **reference goal** sentence for the LLM judge.

In [15]:
# The evaluation dataset: a list of test cases (one dict per test case).
# Each test case contains:
#   "question"             -> what we send to the agent
#   "reference_tool_calls" -> the tool calls a perfect agent WOULD make (ground truth)
#   "reference_goal"       -> plain-English description of a successful outcome (for the LLM judge)
dataset = [
    # ---- Test case 1: multi-tool request (flights + hotel) ----------------------
    {
        "question": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),

        # Two calls expected. Argument values must match EXACTLY
        # (e.g. stars must be the integer 4, not the string "4").
        "reference_tool_calls": [
            ToolCall(
                name="search_flights",
                args={
                    "origin": "Bangalore",
                    "destination": "Dubai",
                },
            ),
            ToolCall(
                name="find_hotels",
                args={
                    "city": "Dubai",
                    "stars": 4,
                },
            ),
        ],

        "reference_goal": """
The agent should provide flight options from Bangalore
to Dubai and 4-star hotel options in Dubai.
""".strip(),
    },

    # ---- Test case 2: single tool (flights only) ---------------------------------
    # A good agent should NOT also call find_hotels or get_weather here.
    {
        "question": """
Find flights from Bangalore to Singapore.
""".strip(),

        "reference_tool_calls": [
            ToolCall(
                name="search_flights",
                args={
                    "origin": "Bangalore",
                    "destination": "Singapore",
                },
            ),
        ],

        "reference_goal": """
The agent should provide flight options
from Bangalore to Singapore.
""".strip(),
    },

    # ---- Test case 3: single tool (hotel only) -----------------------------------
    # Checks that "5-star" in text becomes stars=5 (an integer) in the tool call.
    {
        "question": """
Find a 5-star hotel in Mumbai.
""".strip(),

        "reference_tool_calls": [
            ToolCall(
                name="find_hotels",
                args={
                    "city": "Mumbai",
                    "stars": 5,
                },
            ),
        ],

        "reference_goal": """
The agent should provide 5-star hotel options
in Mumbai.
""".strip(),
    },
]

### Step 10 — Helper functions
`get_message_text` safely turns a message's content into plain text. `extract_tool_calls` collects tool calls from a list of messages.

In [16]:
# Helper: get the plain text out of a LangChain message.
# Why is this needed? A message's `.content` can be:
#   - a plain string                          -> "Here are your flights..."
#   - a LIST of content blocks (newer APIs)   -> [{"type": "text", "text": "..."}, ...]
# This function handles both cases and always returns a string.
def get_message_text(message):

    content = message.content

    # Case 1: already a string -> return as-is
    if isinstance(content, str):
        return content

    # Case 2: list of blocks -> collect the text pieces and join them with newlines
    if isinstance(content, list):

        text_parts = []

        for block in content:

            if isinstance(block, str):
                text_parts.append(block)

            # Dict blocks: keep only those that carry text (skip e.g. reasoning/tool blocks)
            elif isinstance(block, dict) and "text" in block:
                text_parts.append(block["text"])

        return "\n".join(text_parts)

    # Anything else: fall back to its string representation
    return str(content)

In [17]:
# Helper: collect every tool call found in a list of messages (a "trace").
# Only AI messages have a non-empty `tool_calls` attribute; for other messages
# getattr(..., None) safely returns None so they are skipped.
# NOTE: only the commented-out version of run_agent below uses this helper;
#       the active run_agent extracts tool calls itself.
def extract_tool_calls(ragas_trace):

    tool_calls = []

    for message in ragas_trace:

        message_tool_calls = getattr(
            message,
            "tool_calls",
            None,
        )

        if message_tool_calls:

            for tool_call in message_tool_calls:
                tool_calls.append(tool_call)

    return tool_calls


### Step 11 — `run_agent`: run the agent and package the result for RAGAS
This is the heart of the RAGAS integration. The original author first tried RAGAS's built-in converter (`convert_to_ragas_messages`, left commented out at the top), then switched to building the trace **by hand**.

**Design choice worth noticing:** the hand-built trace is just **2 messages**. It has the user question and one AI message that holds the final answer *and all tool calls*. The tool *results* (`ToolMessage`s) are left out. That's enough for the tool-call metrics. The goal-accuracy judge only sees the question, the tool calls and the final answer, not the raw tool outputs.

In [18]:
# ---------------------------------------------------------------------------
# EARLIER VERSION (disabled): used RAGAS's built-in converter
# convert_to_ragas_messages() to turn the full LangChain trace into RAGAS messages.
# It was replaced by the hand-built version below, which gives full control over
# the trace format RAGAS receives.
# ---------------------------------------------------------------------------
# async def run_agent(question):

#     result = await agent.ainvoke(
#         {
#             "messages": [
#                 {
#                     "role": "user",
#                     "content": question,
#                 }
#             ]
#         }
#     )

#     langchain_messages = result["messages"]

#     final_response = get_message_text(
#         langchain_messages[-1]
#     )

#     ragas_trace = convert_to_ragas_messages(
#         langchain_messages
#     )

#     actual_tool_calls = extract_tool_calls(
#         ragas_trace
#     )

#     return {
#         "response": final_response,
#         "ragas_trace": ragas_trace,
#         "actual_tool_calls": actual_tool_calls,
#     }

# RAGAS message classes, renamed with a "Ragas" prefix so they don't get mixed up
# with LangChain's own HumanMessage / AIMessage classes.
from ragas.messages import (
    HumanMessage as RagasHumanMessage,
    AIMessage as RagasAIMessage,
    ToolCall as RagasToolCall,
)


# ACTIVE VERSION: run the agent for one question and return
#   - response          : the agent's final text answer
#   - ragas_trace       : a mini conversation in RAGAS format (input to the metrics)
#   - actual_tool_calls : the list of tool calls the agent really made
async def run_agent(question):

    # `ainvoke` = async invoke. `await` pauses here until the agent finishes its whole
    # loop (LLM -> tools -> LLM -> final answer).
    result = await agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ]
    })

    # Full LangChain trace: [Human, AI(tool_calls), Tool, Tool, ..., AI(final answer)]
    langchain_messages = result["messages"]

    # The LAST message is always the agent's final answer to the user
    final_response = get_message_text(
        langchain_messages[-1]
    )

    # ========================================================
    # EXTRACT TOOL CALLS DIRECTLY FROM LANGCHAIN MESSAGES
    # ========================================================

    actual_tool_calls = []

    # Walk through every message; only AIMessages that requested tools have tool_calls
    for message in langchain_messages:

        tool_calls = getattr(
            message,
            "tool_calls",
            None,
        )

        # Skip messages with no tool calls (Human, Tool and the final AI message)
        if not tool_calls:
            continue

        # LangChain stores each tool call as a dict: {"name": ..., "args": {...}, "id": ...}.
        # Convert each one into a RAGAS ToolCall (only name + args are needed for scoring).
        for tool_call in tool_calls:

            actual_tool_calls.append(
                RagasToolCall(
                    name=tool_call["name"],
                    args=tool_call.get("args", {}),
                )
            )

    # ========================================================
    # CREATE RAGAS TRACE
    # ========================================================

    # A simplified 2-message trace:
    #   1. what the user asked
    #   2. one AI message holding the final answer AND every tool call the agent made
    # Tool outputs (ToolMessages) are left out on purpose; the metrics only need the
    # tool calls, the question and the final answer.
    ragas_trace = [
        RagasHumanMessage(
            content=question,
        ),
        RagasAIMessage(
            content=final_response,
            tool_calls=actual_tool_calls,
        ),
    ]

    return {
        "response": final_response,
        "ragas_trace": ragas_trace,
        "actual_tool_calls": actual_tool_calls,
    }

*(Fully commented-out debug cell.)* If you uncomment it, it prints every message in an agent run with its type and tool calls. That helps when you need to see the raw structure.

In [19]:
# DEBUG HELPER (fully commented out).
# Uncomment to print each message of an agent run: its class name, any tool calls
# and its content. Handy for checking where tool calls live in the trace.

# result = await agent.ainvoke({
#     "messages": [
#         {
#             "role": "user",
#             "content": """
# Plan a Dubai trip from Bangalore.
# Find flights from Bangalore to Dubai
# and find a 4-star hotel in Dubai.
# """.strip(),
#         }
#     ]
# })

# for message in result["messages"]:

#     print(type(message).__name__)

#     print(
#         "Tool Calls:",
#         getattr(message, "tool_calls", None),
#     )

#     print(
#         "Content:",
#         message.content,
#     )

#     print("-" * 100)

### Step 12 — `evaluate_test_case`: score ONE question
Run the agent → score with each of the 3 metrics → return everything as a dictionary (one row of the final results table).

In [20]:
# Evaluate ONE test case (one row of the dataset) and return a result dictionary.
async def evaluate_test_case(row):

    # Unpack the test case
    question = row["question"]
    reference_tool_calls = row["reference_tool_calls"]   # expected tool calls (ground truth)
    reference_goal = row["reference_goal"]               # expected outcome (for LLM judge)

    # ========================================================
    # RUN AGENT
    # ========================================================

    agent_result = await run_agent(question)

    response = agent_result["response"]
    ragas_trace = agent_result["ragas_trace"]
    actual_tool_calls = agent_result["actual_tool_calls"]

    # ========================================================
    # TOOL CALL ACCURACY
    # ========================================================

    # Rule-based: compares tool calls inside `ragas_trace` with `reference_tool_calls`.
    # `.ascore()` is the async scoring method; it returns a MetricResult (score in `.value`).
    tool_call_accuracy_result = await tool_call_accuracy_metric.ascore(
        user_input=ragas_trace,
        reference_tool_calls=reference_tool_calls,
    )

    # ========================================================
    # TOOL CALL F1
    # ========================================================

    # Rule-based: precision/recall of the agent's calls vs the reference calls
    tool_call_f1_result = await tool_call_f1_metric.ascore(
        user_input=ragas_trace,
        reference_tool_calls=reference_tool_calls,
    )

    # ========================================================
    # AGENT GOAL ACCURACY
    # ========================================================

    # LLM-judge: the judge model reads the trace and decides whether the agent's
    # outcome matches `reference_goal` (1.0 = yes, 0.0 = no). This costs an API call.
    agent_goal_accuracy_result = await agent_goal_accuracy_metric.ascore(
        user_input=ragas_trace,
        reference=reference_goal,
    )

    # ========================================================
    # FORMAT TOOL CALLS
    # ========================================================

    # Turn ToolCall objects into readable strings, e.g.
    #   "search_flights({'origin': 'Bangalore', 'destination': 'Dubai'})"
    actual_tool_calls_text = [
        f"{tool_call.name}({tool_call.args})"
        for tool_call in actual_tool_calls
    ]

    reference_tool_calls_text = [
        f"{tool_call.name}({tool_call.args})"
        for tool_call in reference_tool_calls
    ]

    # ========================================================
    # RETURN RESULT
    # ========================================================

    # One dict = one row in the results DataFrame. `.value` holds the numeric score.
    return {
        "question": question,
        "response": response,
        "actual_tool_calls": actual_tool_calls_text,
        "reference_tool_calls": reference_tool_calls_text,
        "tool_call_accuracy": tool_call_accuracy_result.value,
        "tool_call_f1": tool_call_f1_result.value,
        "agent_goal_accuracy": agent_goal_accuracy_result.value,
    }


### Step 13 — `run_evaluation`: score ALL questions
A simple loop over the dataset. Test cases run **one after another** (sequentially). That's slower than running them in parallel, but easier to follow and gentler on API rate limits.

In [21]:
# Evaluate EVERY test case in the dataset, one after another, and collect the results.
async def run_evaluation():

    results = []

    # enumerate(..., start=1) gives a human-friendly counter: 1, 2, 3
    for index, row in enumerate(dataset, start=1):

        # Progress banner, e.g. "Running Test Case 1/3"
        print("\n" + "=" * 100)
        print(f"Running Test Case {index}/{len(dataset)}")
        print("=" * 100)

        # Run agent + all metrics for this test case
        result = await evaluate_test_case(row)

        results.append(result)

    # List of dicts -> easy to turn into a pandas DataFrame
    return results

### Step 14 — `main`: run everything and report
Prints a results table, the average score per metric, saves a CSV and then prints each test case in detail (question, answer, expected vs actual tool calls, scores).

In [22]:
# Main entry point: run the evaluation, then report and save the results.
async def main():

    print("\n" + "=" * 100)
    print("RAGAS AGENTIC AI EVALUATION")
    print("=" * 100)

    # ========================================================
    # RUN EVALUATION
    # ========================================================

    results = await run_evaluation()

    # ========================================================
    # CONVERT TO DATAFRAME
    # ========================================================

    # Each dict becomes a row; dict keys become column names
    df = pd.DataFrame(results)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    # Show only the question and the 3 scores in the compact table
    columns_to_show = [
        "question",
        "tool_call_accuracy",
        "tool_call_f1",
        "agent_goal_accuracy",
    ]

    # to_string(index=False) prints the table without the 0/1/2 row numbers
    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # SUMMARY
    # ========================================================

    # Average of each metric across all test cases (the overall "grade")
    average_tool_call_accuracy = df["tool_call_accuracy"].mean()
    average_tool_call_f1 = df["tool_call_f1"].mean()
    average_agent_goal_accuracy = df["agent_goal_accuracy"].mean()

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    # :.2f -> format the number with 2 decimal places
    print(f"Average Tool Call Accuracy : {average_tool_call_accuracy:.2f}")
    print(f"Average Tool Call F1       : {average_tool_call_f1:.2f}")
    print(f"Average Agent Goal Accuracy: {average_agent_goal_accuracy:.2f}")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    # Save the full results (all columns) so you can open them in Excel or compare runs
    output_file = "ragas_agent_evaluation_results.csv"

    df.to_csv(
        output_file,
        index=False,
    )

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT EACH TEST CASE
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED AGENT RESULTS")
    print("=" * 100)

    # iterrows() yields (row_index, row) pairs; we ignore the index with "_"
    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nAGENT RESPONSE:")
        print(result_row["response"])

        # Put expected and actual side by side so mismatches are easy to spot
        print("\nEXPECTED TOOL CALLS:")
        print(result_row["reference_tool_calls"])

        print("\nACTUAL TOOL CALLS:")
        print(result_row["actual_tool_calls"])

        print("\nTOOL CALL ACCURACY:")
        print(result_row["tool_call_accuracy"])

        print("\nTOOL CALL F1:")
        print(result_row["tool_call_f1"])

        print("\nAGENT GOAL ACCURACY:")
        print(result_row["agent_goal_accuracy"])

        print("\n" + "-" * 100)

### Step 15 — Run it!
`await main()` works directly in Jupyter because Jupyter already runs an async event loop. (In a `.py` script, use `asyncio.run(main())`.)

**Result:** all three test cases scored **1.0 on every RAGAS metric**. The agent chose the right tools, with the right arguments, and achieved each goal.

In [23]:
# Run the whole RAGAS evaluation.
# Top-level `await` works in Jupyter (it already has an event loop running).
# In a plain Python script you would use: asyncio.run(main())
await main()


RAGAS AGENTIC AI EVALUATION

Running Test Case 1/3

Running Test Case 2/3

Running Test Case 3/3

DETAILED RESULTS
                                                                                                  question  tool_call_accuracy  tool_call_f1  agent_goal_accuracy
Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.                 1.0           1.0                  1.0
                                                                 Find flights from Bangalore to Singapore.                 1.0           1.0                  1.0
                                                                            Find a 5-star hotel in Mumbai.                 1.0           1.0                  1.0

SUMMARY
Average Tool Call Accuracy : 1.00
Average Tool Call F1       : 1.00
Average Agent Goal Accuracy: 1.00

Results saved to:
ragas_agent_evaluation_results.csv

DETAILED AGENT RESULTS

QUESTION:
Plan a Dubai trip from Bangalore.
Fi

---
# Part B — Agent evaluation with DeepEval

Now we evaluate **the same kind of agent on the same three questions** with a different library, DeepEval. The pipeline has the same shape (`run_agent` → `evaluate_test_case` → `run_evaluation` → `main`), but uses DeepEval's own data types and metrics.

> ⚠️ Several names (`ToolCall`, `llm`, `agent`, `dataset`, `run_agent`, `evaluate_test_case`, `main`) are **redefined** below and replace the RAGAS versions from Part A.

### Step B1 — DeepEval imports
- `LLMTestCase`: a **single-turn** test case (one input → one output + tools used)
- `ConversationalTestCase` + `Turn`: a **multi-turn** conversation, needed by `GoalAccuracyMetric`
- `ToolCall`: DeepEval's tool-call type (uses `input_parameters` where RAGAS uses `args`)

In [24]:
# Re-import the LangChain pieces (harmless if already imported in Part A)
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# DeepEval test-case data structures
from deepeval.test_case import (
    LLMTestCase,              # single-turn: input, actual_output, tools_called, expected_tools
    ConversationalTestCase,   # multi-turn: a list of Turns (needed by GoalAccuracyMetric)
    Turn,                     # one message in a conversation (role + content + tools_called)
    ToolCall,                 # DeepEval's tool call: name + input_parameters
                              # NOTE: this REPLACES the RAGAS `ToolCall` imported in Part A
)

# DeepEval agent metrics
from deepeval.metrics import (
    ToolCorrectnessMetric,       # rule-based: were the expected tools called?
    ArgumentCorrectnessMetric,   # LLM-judge: were the tool arguments correct for the input?
    GoalAccuracyMetric,          # LLM-judge: did the agent complete the task, and plan well?
)


### Step B2 — The same three mock tools
These are identical to Part A. They're redefined so this half of the notebook can run on its own.

In [25]:
# The same three MOCK tools as in Part A (see the Part A comments for details).
# Redefined here so the DeepEval half can run on its own.

# Tool 1: fake flight search
@tool
def search_flights(origin: str, destination: str) -> str:
    """
    Search flights between an origin and destination city.
    """

    return f"""
Available flights from {origin} to {destination}:

1. Emirates EK569
   Price: ₹28,000
   Type: Non-stop

2. Air India AI995
   Price: ₹24,500
   Type: 1 stop
""".strip()


# Tool 2: fake hotel search (stars must be an integer)
@tool
def find_hotels(city: str, stars: int) -> str:
    """
    Find hotels in a city based on star rating.
    """

    return f"""
Available {stars}-star hotels in {city}:

1. Novotel {city}
   Price: ₹8,500 per night

2. Hilton Garden Inn {city}
   Price: ₹9,200 per night
""".strip()


# Tool 3: fake weather lookup. A "distractor" that should NOT be called in our tests.
@tool
def get_weather(city: str) -> str:
    """
    Get weather information for a city.
    """

    return f"""
Weather in {city}:

Temperature: 30°C
Condition: Sunny
""".strip()


### Step B3 — A different agent model
Part B uses `gpt-5.4-mini` with no `temperature` set (so the default is used). Results may vary a little from run to run.

In [26]:
# The agent's model for Part B. No temperature is set, so the model's default is used
# (outputs may vary slightly between runs).
llm = ChatOpenAI(
    model="gpt-5.4-mini",
)

### Step B4 — Build the agent (same tools, same rules)

In [27]:
# Build the DeepEval-section agent: same tools, same rules as Part A, new model.
agent = create_agent(
    model=llm,
    tools=[
        search_flights,
        find_hotels,
        get_weather,
    ],
    # Same behavioural rules as before (choose the right tool, no extra calls,
    # exact city names, clear final answer)
    system_prompt="""
You are a travel planning assistant.

Use the available tools whenever the user asks for
flight, hotel, or weather information.

Rules:

1. Use search_flights for flight requests.
2. Use find_hotels for hotel requests.
3. Use get_weather only when weather is requested.
4. Do not call unnecessary tools.
5. Use city names exactly as provided by the user.
6. After using the tools, provide a clear final answer.
""",
)


### Step B5 — The DeepEval test dataset
Same questions and same expected tools. The differences from Part A: DeepEval's `ToolCall` uses **`input_parameters`**, and there's **no reference goal**, because DeepEval's goal metric judges the conversation without one.

In [28]:
# DeepEval dataset: same 3 questions as Part A.
# Differences from the RAGAS dataset:
#   - key "expected_tools" (instead of "reference_tool_calls")
#   - DeepEval ToolCall uses `input_parameters=` (instead of RAGAS's `args=`)
#   - no "reference_goal": DeepEval's GoalAccuracyMetric judges without one
dataset = [
    # Test case 1: needs BOTH flight and hotel tools
    {
        "question": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="search_flights",
                input_parameters={
                    "origin": "Bangalore",
                    "destination": "Dubai",
                },
            ),
            ToolCall(
                name="find_hotels",
                input_parameters={
                    "city": "Dubai",
                    "stars": 4,
                },
            ),
        ],
    },

    # Test case 2: flights only
    {
        "question": """
Find flights from Bangalore to Singapore.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="search_flights",
                input_parameters={
                    "origin": "Bangalore",
                    "destination": "Singapore",
                },
            ),
        ],
    },

    # Test case 3: hotel only
    {
        "question": """
Find a 5-star hotel in Mumbai.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="find_hotels",
                input_parameters={
                    "city": "Mumbai",
                    "stars": 5,
                },
            ),
        ],
    },
]


### Step B6 — Helper functions
Same text helper as before. The tool-call extractor now builds **DeepEval** `ToolCall` objects directly from LangChain messages, and `format_tool_calls` makes them human-readable for the report.

In [29]:
# Same helper as in Part A: return a message's content as plain text,
# whether it is stored as a string or as a list of content blocks.
def get_message_text(message):

    content = message.content

    # Plain string -> return directly
    if isinstance(content, str):
        return content

    # List of blocks -> join the text pieces
    if isinstance(content, list):

        text_parts = []

        for block in content:

            if isinstance(block, str):
                text_parts.append(block)

            elif isinstance(block, dict) and "text" in block:
                text_parts.append(block["text"])

        return "\n".join(text_parts)

    # Fallback for any other type
    return str(content)


In [30]:
# Collect all tool calls from the agent's LangChain messages and convert them
# into DeepEval ToolCall objects (name + input_parameters).
def extract_tool_calls(messages):

    tool_calls = []

    for message in messages:

        # Only AI messages that requested tools have a non-empty `tool_calls`
        message_tool_calls = getattr(message, "tool_calls", None)

        if not message_tool_calls:
            continue

        for tool_call in message_tool_calls:

            # LangChain dict {"name", "args", "id"} -> DeepEval ToolCall
            tool_calls.append(
                ToolCall(
                    name=tool_call["name"],
                    input_parameters=tool_call.get("args", {}),
                )
            )

    return tool_calls

In [31]:
# Turn DeepEval ToolCall objects into readable strings for the report/CSV, e.g.
#   "find_hotels({'city': 'Mumbai', 'stars': 5})"
def format_tool_calls(tool_calls):

    return [
        f"{tool_call.name}({tool_call.input_parameters})"
        for tool_call in tool_calls
    ]


### Step B7 — `run_agent` (DeepEval version)
Simpler than the RAGAS version, because DeepEval doesn't need a special message-trace format. We only need the final text and the list of tool calls.

In [32]:
# Run the agent for one question and return the final answer + the tool calls made.
# (No special trace format is needed for DeepEval, unlike RAGAS.)
async def run_agent(question):

    # Async agent run: LLM -> tool(s) -> LLM ... -> final answer
    result = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    # Full message trace of the run
    messages = result["messages"]

    # Last message = final answer shown to the user
    response = get_message_text(messages[-1])

    # Every tool call made during the run, as DeepEval ToolCall objects
    actual_tool_calls = extract_tool_calls(messages)

    return {
        "response": response,
        "actual_tool_calls": actual_tool_calls,
    }


### Step B8 — `evaluate_test_case`: score ONE question with DeepEval
1. Run the agent.
2. Wrap the result in an **`LLMTestCase`** and score it with **ToolCorrectness** (rule-based) and **ArgumentCorrectness** (LLM judge).
3. Wrap the same result in a **`ConversationalTestCase`** (user turn + assistant turn) and score it with **GoalAccuracy** (LLM judge).
4. Return the scores **and the judges' written reasons**.

`threshold=0.7` sets the pass/fail line. `include_reason=True` asks the metric to explain its score. `async_mode=False` makes `.measure()` run synchronously, one step at a time.

In [33]:
# Evaluate ONE test case with the three DeepEval metrics.
async def evaluate_test_case(row):

    question = row["question"]
    expected_tools = row["expected_tools"]   # ground-truth tool calls

    # ========================================================
    # RUN AGENT
    # ========================================================

    agent_result = await run_agent(question)

    response = agent_result["response"]
    actual_tool_calls = agent_result["actual_tool_calls"]

    # ========================================================
    # CREATE SINGLE-TURN TEST CASE
    # ========================================================

    # LLMTestCase bundles everything about one interaction:
    #   input          -> what the user asked
    #   actual_output  -> what the agent answered
    #   tools_called   -> tools the agent REALLY called
    #   expected_tools -> tools it SHOULD have called
    agent_test_case = LLMTestCase(
        input=question,
        actual_output=response,
        tools_called=actual_tool_calls,
        expected_tools=expected_tools,
    )

    # ========================================================
    # CREATE TOOL CORRECTNESS METRIC
    # ========================================================

    # Rule-based (no LLM): compares tools_called vs expected_tools by tool name.
    # threshold=0.7     -> score >= 0.7 counts as "pass"
    # include_reason    -> also produce a human-readable explanation
    tool_correctness_metric = ToolCorrectnessMetric(
        threshold=0.7,
        include_reason=True,
    )

    # ========================================================
    # CREATE ARGUMENT CORRECTNESS METRIC
    # ========================================================

    # LLM-judge (gpt-5.4): given the user's input, were the ARGUMENTS passed to
    # each tool correct? async_mode=False -> .measure() runs its steps synchronously.
    argument_correctness_metric = ArgumentCorrectnessMetric(
        threshold=0.7,
        model="gpt-5.4",
        include_reason=True,
        async_mode=False,
    )

    # ========================================================
    # TOOL CORRECTNESS EVALUATION
    # ========================================================

    # .measure() computes the score and stores it on the metric object
    # (read it afterwards with .score and .reason)
    tool_correctness_metric.measure(agent_test_case)

    # ========================================================
    # ARGUMENT CORRECTNESS EVALUATION
    # ========================================================

    argument_correctness_metric.measure(agent_test_case)

    # ========================================================
    # CREATE CONVERSATIONAL TEST CASE
    # ========================================================

    # GoalAccuracyMetric works on CONVERSATIONS, so we rebuild the same interaction
    # as two turns: the user's question, then the assistant's answer + tools used.
    conversation_test_case = ConversationalTestCase(
        turns=[
            Turn(
                role="user",
                content=question,
            ),
            Turn(
                role="assistant",
                content=response,
                tools_called=actual_tool_calls,
            ),
        ]
    )

    # ========================================================
    # CREATE GOAL ACCURACY METRIC
    # ========================================================

    # LLM-judge (gpt-5.4): scores (a) how well the task was completed and
    # (b) how good the agent's plan/approach was, then combines them.
    # Usually stricter than RAGAS's binary goal check.
    goal_accuracy_metric = GoalAccuracyMetric(
        threshold=0.7,
        model="gpt-5.4",
        include_reason=True,
        async_mode=False,
    )

    # ========================================================
    # GOAL ACCURACY EVALUATION
    # ========================================================

    goal_accuracy_metric.measure(conversation_test_case)

    # ========================================================
    # RETURN RESULT
    # ========================================================

    # One row of the results table: readable tool calls + each metric's score AND reason
    return {
        "question": question,
        "response": response,
        "expected_tools": format_tool_calls(expected_tools),
        "actual_tool_calls": format_tool_calls(actual_tool_calls),

        "tool_correctness": tool_correctness_metric.score,
        "tool_correctness_reason": tool_correctness_metric.reason,

        "argument_correctness": argument_correctness_metric.score,
        "argument_correctness_reason": argument_correctness_metric.reason,

        "goal_accuracy": goal_accuracy_metric.score,
        "goal_accuracy_reason": goal_accuracy_metric.reason,
    }


### Step B9 — `run_evaluation`: loop over all test cases (same as Part A)

In [34]:
# Same loop as Part A: evaluate each test case in order and collect the result dicts.
# (Uses the DeepEval `dataset` and `evaluate_test_case` defined above.)
async def run_evaluation():

    results = []

    for index, row in enumerate(dataset, start=1):

        # Progress banner
        print("\n" + "=" * 100)
        print(f"Running Test Case {index}/{len(dataset)}")
        print("=" * 100)

        result = await evaluate_test_case(row)

        results.append(result)

    return results

### Step B10 — `main`: run, summarise, save CSV and print the reasons

In [35]:
# Main entry point for the DeepEval run: evaluate, summarise, save and explain.
async def main():

    print("\n" + "=" * 100)
    print("DEEPEVAL AGENTIC AI EVALUATION")
    print("=" * 100)

    # ========================================================
    # RUN EVALUATION
    # ========================================================

    results = await run_evaluation()

    # ========================================================
    # CONVERT TO PANDAS
    # ========================================================

    # One row per test case; columns = the keys returned by evaluate_test_case
    df = pd.DataFrame(results)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    # Compact score table (question + 3 scores)
    columns_to_show = [
        "question",
        "tool_correctness",
        "argument_correctness",
        "goal_accuracy",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # CALCULATE AVERAGE SCORES
    # ========================================================

    # Mean score per metric across all test cases
    average_tool_correctness = df["tool_correctness"].mean()
    average_argument_correctness = df["argument_correctness"].mean()
    average_goal_accuracy = df["goal_accuracy"].mean()

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Average Tool Correctness     : {average_tool_correctness:.2f}")
    print(f"Average Argument Correctness : {average_argument_correctness:.2f}")
    print(f"Average Goal Accuracy        : {average_goal_accuracy:.2f}")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    # CSV includes the judges' written reasons, which are useful for later analysis
    output_file = "deepeval_agent_evaluation_results.csv"

    df.to_csv(output_file, index=False)

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED AGENT EVALUATION REASONS")
    print("=" * 100)

    # For each test case, print the answer, expected vs actual tools,
    # and each metric's score WITH its explanation
    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nAGENT RESPONSE:")
        print(result_row["response"])

        print("\nEXPECTED TOOLS:")
        print(result_row["expected_tools"])

        print("\nACTUAL TOOL CALLS:")
        print(result_row["actual_tool_calls"])

        print("\nTOOL CORRECTNESS:")
        print(result_row["tool_correctness"])

        print("\nTOOL CORRECTNESS REASON:")
        print(result_row["tool_correctness_reason"])

        print("\nARGUMENT CORRECTNESS:")
        print(result_row["argument_correctness"])

        print("\nARGUMENT CORRECTNESS REASON:")
        print(result_row["argument_correctness_reason"])

        print("\nGOAL ACCURACY:")
        print(result_row["goal_accuracy"])

        print("\nGOAL ACCURACY REASON:")
        print(result_row["goal_accuracy_reason"])

        print("\n" + "-" * 100)

### Step B11 — Run it!

**Result:** Tool Correctness = **1.0** and Argument Correctness = **1.0** for every case, which matches RAGAS. **Goal Accuracy averages only 0.58** (0.75 / 0.375 / 0.625). Read the *GOAL ACCURACY REASON* lines. DeepEval's judge scores **task completion and planning separately** and averages them (e.g. 0.5 task + 1.0 plan → 0.75). It also wanted extra details the tools never provided (dates, times, availability). Only test case 1 clears the 0.7 threshold.

**Lesson:** a perfect tool-calling score doesn't guarantee a "perfect" agent. LLM-judge metrics measure *user satisfaction*, and different frameworks set the bar differently. Check the reasons, and calibrate thresholds for your use case.

(The `ipywidgets` warnings are harmless. They only mean the progress bars can't render as widgets.)

In [36]:
# Run the whole DeepEval evaluation (top-level await works in Jupyter).
await main()


DEEPEVAL AGENTIC AI EVALUATION

Running Test Case 1/3


/Users/manny/Documents/git_repos/Full-stack-GenAI/env/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


Running Test Case 2/3


/Users/manny/Documents/git_repos/Full-stack-GenAI/env/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


Running Test Case 3/3


/Users/manny/Documents/git_repos/Full-stack-GenAI/env/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


DETAILED RESULTS
                                                                                                  question  tool_correctness  argument_correctness  goal_accuracy
Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.               1.0                   1.0          0.750
                                                                 Find flights from Bangalore to Singapore.               1.0                   1.0          0.375
                                                                            Find a 5-star hotel in Mumbai.               1.0                   1.0          0.625

SUMMARY
Average Tool Correctness     : 1.00
Average Argument Correctness : 1.00
Average Goal Accuracy        : 0.58

Results saved to:
deepeval_agent_evaluation_results.csv

DETAILED AGENT EVALUATION REASONS

QUESTION:
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.

AGENT RE

---
## Key takeaways

1. **Agent evaluation has layers.** Check *tool choice*, *tool arguments* and *final outcome* separately. An agent can get the first two perfect and still fall short on the third.
2. **Rule-based metrics** (ToolCallAccuracy, ToolCallF1, ToolCorrectness) are cheap, fast and deterministic. Use them for regression tests in CI.
3. **LLM-judge metrics** (AgentGoalAccuracy, ArgumentCorrectness, GoalAccuracy) catch quality issues that exact matching can't, but they cost money, can vary from run to run and depend on how strict the judge is. **Always read the reasons.**
4. **Frameworks disagree.** RAGAS gave goal accuracy 1.00. DeepEval gave 0.58 for similar behaviour. Choose metrics and thresholds that match what *your* users actually need.

### Ideas to experiment with
- Add a test case that asks about weather, and one that should *not* trigger any tool. Do the metrics catch mistakes?
- Set `strict_order=True` in `ToolCallAccuracy` and see whether scores change.
- Make the mock tools return dates and times. Does DeepEval's goal accuracy go up?
- Swap the agent model (e.g. a smaller model) and compare the CSV files from different runs.